# 01. Advanced Agent Capability 설계 실습

> Part 1 · 50분 · 핵심 질문: **능력은 무엇으로 이루어지고, 판단은 어디에 두는가?**

## 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 1 | 준비와 한 가지 사례 소개 | 5 |
| 2 | 능력 = 지침(Skill) + 도구 + 실행 제어 + 상태 | 12 |
| 3 | `create_deep_agent`의 기본 도구와 `build_agent`가 바꾸는 것 | 8 |
| 4 | 판단은 어디에 두는가: 코드 규칙, 판단 모델(Jev), 생성(LLM) | 10 |
| 5 | 실제 실행 1회와 읽을 수 있는 트레이스 | 10 |
| 6 | 병목 귀속 미리보기와 정리 | 5 |

## 이 노트북에서 하는 일

과정 전체(00~07)는 하나의 사례를 끝까지 끌고 갑니다. **요구사항 인터뷰 MCP 서버**를 **Deep Agent**가 부리는 사례입니다.

- 서버는 도구 7개를 공개합니다: `start_interview`, `record_answer`, `get_session`, `freeze_seed`, `list_evidence`, `read_evidence`, `save_evidence`.
- 세션은 `draft` -> `ready` -> `frozen` 순서로 상태가 바뀝니다(`cap_lab/server/workflow.py:82`).
- 에이전트는 `create_deep_agent`에 ChatAnthropic `claude-sonnet-5-5`(effort `medium`)를 붙여 만듭니다. 조립 코드는 `cap_lab/agent.py:89`의 `build_agent`입니다.
- 병목으로 측정된 곳의 **판단(Decision)** 은 Jev(TypeSafe Decision API)를 거치고, 측정 기록은 Langfuse에 남깁니다. 그 측정은 07에서 합니다. 이 노트북은 **어디에 판단을 두는지**를 눈으로 확인하는 데까지만 갑니다.

이 과정의 모든 코드는 제공됩니다. 점수를 매기는 과제는 없고, 셀을 돌려 보며 **자기 말로 설명할 수 있는지**가 목표입니다.

## 호출 규모

유료 호출이 있는 셀은 실행 직전에 `announce(...)`가 규모를 한 줄로 알립니다. 이 노트북 전체의 상한 추정은 아래와 같습니다. 직접 해보기의 셀은 `RUN_EXERCISE = False`일 때 아무것도 호출하지 않습니다.

| 셀 | 하는 일 | Claude | Jev | 비고 |
|---|---|---|---|---|
| 3절 조립 셀 둘 | `build_agent`로 조립만 한다 | 0회 | 0회 | MCP 서버 자식 프로세스만 띄운다 |
| 4절 Jev 셀 | 완료 기준 3개의 testability 판정 | 0회 | 최대 3회 | |
| 5절 실행 셀 | dev-01 사례 1건 | 약 5~9회 | 0회 | 게이트를 끈 기준 변형 |
| 직접 해보기 | `RUN_EXERCISE = True`일 때만 | 최대 약 9회 | 최대 3회 | 기본은 꺼짐 |
| 5절 선택 셀 | `RUN_OPTIONAL = True`일 때만 | 약 5~9회 | 0회 | 기본은 꺼짐 |

Claude 호출 수는 `docs/DESIGN.md` 8절의 사례당 약 5~9회 추정입니다. 한 번 실행한 결과는 **1회 관찰**입니다. 평균이나 개선의 근거로 쓰지 않습니다. 반복과 비교 방법은 07에서 다룹니다.

## 선행 조건

- 저장소 루트에서 `uv sync --locked`를 마쳤고, 이 노트북의 작업 폴더는 `Day04_new`입니다. 그래서 `import cap_lab`이 됩니다.
- `Day04_new/.env`에 `ANTHROPIC_API_KEY`와 Decision API 키(이 과정에서는 `TYPESAFE_API_KEY`)가 필요합니다. 키 값은 어디에도 출력하지 않습니다. 설정됨/미설정만 확인합니다.

## 1. 준비

다음 셀은 이 노트북이 쓰는 모든 import와 두 개의 스위치를 한곳에 모읍니다.

- `RUN_EXERCISE`: 직접 해보기 중 돈이 드는 셀을 켜는 스위치입니다. 기본 `False`.
- `RUN_OPTIONAL`: 시간이 남을 때만 돌리는 무거운 선택 셀을 켜는 스위치입니다. 기본 `False`.

스위치를 `True`로 바꾸는 것은 **본인이 크레딧을 쓰기로 결정하는 일**입니다. 바꾸기 전에 셀 안의 `announce(...)` 문구를 읽으세요.

`cap_lab.notebook`은 수업용 도우미 모듈입니다(`cap_lab/notebook.py`). 이 노트북에서는 `announce`(규모 고지), `connect_server`(서버를 stdio로 띄워 연결), `new_workdir`(작업 폴더 만들기), `show_events`(서버가 남긴 근거 이벤트 표), `show_table`(표 출력), `key_table`(키 상태)을 씁니다.

In [ ]:
# [따라하기] import와 스위치, 키 상태 확인 (값은 출력하지 않는다)
import asyncio
import inspect
import json
import math
import sqlite3
import tempfile
from contextlib import closing
from pathlib import Path

import yaml
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver

import cap_lab.agent as agent_module
import cap_lab.middleware as middleware_module
from cap_lab.agent import SKILLS_DIR, build_agent, server_argv
from cap_lab.decisions import DecisionError, make_decider
from cap_lab.experiment import Variant, load_cases, run_case
from cap_lab.notebook import announce, connect_server, key_table, new_workdir, show_events, show_table
from cap_lab.scorers import STAGE_OF, STAGE_ORDER, Score, first_failure, score_run
from cap_lab.server.gate import TESTABILITY, GateDecision
from cap_lab.server.server import FaultPlan, business_failure
from cap_lab.server.workflow import WorkflowError
from cap_lab.settings import ROOT, load_settings, make_model

# 돈이 드는 선택 셀의 스위치. 켜기 전에 해당 셀의 announce 문구를 읽는다.
RUN_EXERCISE = False
RUN_OPTIONAL = False

settings = load_settings()
print("작업 폴더 기준(ROOT):", ROOT.name)
print("모델:", settings.model, "| effort:", settings.effort)
key_table(settings)

**관찰할 것.** `ANTHROPIC_API_KEY`와 `TYPESAFE_API_KEY`가 `설정됨`인지 봅니다. 미설정이면 이 노트북의 유료 셀은 멈춥니다. 키를 코드나 노트북에 직접 적지 말고 `Day04_new/.env.example`을 보고 `.env`에 직접 채우세요. `LANGFUSE_*` 키는 이 노트북에서 쓰지 않습니다. 측정 기록은 07에서 남깁니다.

## 2. 능력 = 지침(Skill) + 도구 + 실행 제어 + 상태

"에이전트가 이걸 할 수 있다"는 말은 모델이 똑똑하다는 뜻이 아닙니다. 모델 바깥에 있는 **네 가지가 맞물려 있다**는 뜻입니다.

| 구성 요소 | 하는 일 | 이 사례에서 어디에 있나 |
|---|---|---|
| 지침(Skill) | 언제 무엇을 어떤 순서로 하라고 알려 준다 | `skills/requirements-interview/SKILL.md` |
| 도구 | 모델이 바깥 세계에 일을 시키는 손 | MCP 서버의 도구 7개(`cap_lab/server/server.py`), Deep Agent의 파일 도구 4개 |
| 실행 제어 | 도구 호출 앞뒤에서 막거나 숨기거나 고치는 코드 | 목표 유지 게이트 `GoalGuard`(`cap_lab/middleware.py:52`), 도구 공개 지연(`cap_lab/agent.py:148`), 쓰기 금지(`cap_lab/agent.py:137`) |
| 상태 | 한 번의 호출이 끝나도 남는 것 | 서버의 SQLite와 근거 파일, 에이전트의 체크포인터 |

네 가지 중 하나가 빠지면 어떻게 되는지를 아래에서 직접 봅니다. 먼저 지침부터입니다.

### 2.1 지침: Skill 파일

Skill은 폴더 하나와 `SKILL.md` 파일 하나입니다. 머리말(frontmatter)의 `description`은 **항상** 모델의 시스템 프롬프트에 올라가고, 본문은 모델이 `read_file`로 읽을 때만 들어옵니다. 이것을 점진적 공개(progressive disclosure)라고 부릅니다(`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:919-920`의 클래스 설명). SKILL.md의 형식은 별도 자료에서 가르치므로 여기서는 머리말에서 **세 가지만** 봅니다: `description`, `metadata.include_tools`, 그리고 본문 8번의 오류 처리 규칙.

**먼저 예측하세요.** 머리말의 `include_tools`에는 몇 개의 이름이 있을까요? 그 이름들은 모델이 처음부터 볼 수 있는 도구일까요, 읽은 뒤에 보이는 도구일까요? 다음 셀을 돌리기 전에 한 줄로 적어 보세요.

In [ ]:
# [따라하기] SKILL.md 머리말을 줄 번호와 함께 읽는다.
skill_path = SKILLS_DIR / "requirements-interview" / "SKILL.md"
skill_text = skill_path.read_text(encoding="utf-8")
skill_lines = skill_text.splitlines()

print("Skill 파일:", skill_path.relative_to(ROOT))
print("--- 머리말 (1~9줄) ---")
for number, line in enumerate(skill_lines[:9], start=1):
    print(f"{number:>3}  {line}")

# frontmatter는 첫 '---'와 둘째 '---' 사이의 YAML이다.
front = yaml.safe_load(skill_text.split("---")[1])
include_tools = front["metadata"]["include_tools"].split()
print()
print("name:", front["name"])
print("description 글자 수:", len(front["description"]), "(deepagents가 허용하는 최대는 1024자, .venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:173)")
print("include_tools 개수:", len(include_tools))
print("include_tools:", include_tools)
print()
print("--- 본문 8번: 오류 코드별 행동 (41~47줄) ---")
for number in range(41, 48):
    print(f"{number:>3}  {skill_lines[number - 1]}")

SKILL.md 본문 8번에 주목하세요. `goal_changed`, `permission_denied`, `temporarily_unavailable`, `evidence_publish_failed` 같은 **오류 코드마다 다음 행동**이 적혀 있습니다. 이 규칙이 왜 지침 문장으로도, 코드로도 쓰일 수 있는지는 4절에서 다시 봅니다.

### 2.2 도구: 서버가 공개하는 것

서버는 `cap_lab/server/server.py`입니다. 다음 셀은 서버를 자식 프로세스로 띄워 `tools/list`만 호출합니다. **모델을 부르지 않습니다.** 서버가 도구마다 붙인 힌트(`ToolAnnotations`)도 같이 봅니다. 힌트는 클라이언트에게 주는 **참고 정보일 뿐 강제가 아닙니다**(`cap_lab/server/server.py:48-50`의 주석).

In [ ]:
# [따라하기] 서버를 띄워 도구 목록만 읽는다 (모델 호출 없음)
list_dir = new_workdir("nb01-list")
async with connect_server(list_dir) as client:
    listing = await client.list_tools()

tool_rows = []
for tool in listing.tools:
    hints = tool.annotations
    tool_rows.append({
        "도구": tool.name,
        "읽기 전용": hints.read_only_hint,
        "같은 호출 반복해도 같은 결과(멱등)": hints.idempotent_hint,
        "인자": ", ".join(tool.input_schema.get("properties", {})),
    })
show_table(tool_rows)
print("도구 개수:", len(listing.tools))
print("멱등 칸이 None인 도구는 서버가 그 힌트를 적지 않은 것이다(읽기 전용 도구 둘).")

### 2.3 상태: SQLite, 근거 파일, 그리고 체크포인터

상태는 두 곳에 있고 수명이 다릅니다.

1. **서버 쪽**: SQLite `events` 표(`cap_lab/server/workflow.py:103-110`)와 근거 폴더의 JSON 파일. 에이전트 프로세스가 죽어도 남습니다. 세션의 현재 상태는 저장된 이벤트를 처음부터 다시 읽어 만듭니다(`replay`, `cap_lab/server/workflow.py:35`).
2. **에이전트 쪽**: LangGraph 체크포인터. 실험 실행기는 `InMemorySaver`를 쓰고 `thread_id`로 대화를 구분합니다(`cap_lab/experiment.py:211`, `:214`). 프로세스가 끝나면 사라집니다. 서버가 사용자에게 되묻는 MRTR을 재개하려면 체크포인터와 `thread_id`가 필요합니다(`cap_lab/agent.py:104`).

다음 셀은 **모델 없이** 도구 4개를 직접 불러 세션이 `draft` -> `ready` -> `frozen`으로 바뀌는 모습과, 그때 SQLite와 근거 폴더에 무엇이 쌓이는지 보여 줍니다. `ANSWER_BOTH`를 `False`로 바꾸면 완료 기준을 적지 않고 확정을 시도합니다. 어떤 오류 코드가 돌아올지 먼저 예측해 보세요.

In [ ]:
# [바꿔보기] ANSWER_BOTH를 False로 바꾸고 다시 실행해 본다.
ANSWER_BOTH = True   # True: 제약과 완료 기준을 모두 기록한다. False: 제약만 기록하고 확정을 시도한다.

state_dir = new_workdir("nb01-state")
history = []   # (호출, 결과 status 또는 error)

async with connect_server(state_dir) as client:
    started = await client.call_tool("start_interview", {"goal": "팀 비품 대여 현황 도구"})
    session_id = started.structured_content["session_id"]
    history.append(("start_interview", started.structured_content["status"]))

    first = await client.call_tool(
        "record_answer", {"session_id": session_id, "field": "constraints", "answer": "없음"})
    history.append(("record_answer(constraints)", first.structured_content["status"]))

    if ANSWER_BOTH:
        second = await client.call_tool(
            "record_answer",
            {"session_id": session_id, "field": "acceptance_criteria", "answer": "pytest가 실패 0건으로 끝난다"})
        history.append(("record_answer(acceptance_criteria)", second.structured_content["status"]))

    frozen = await client.call_tool("freeze_seed", {"session_id": session_id})
    result = frozen.structured_content
    history.append(("freeze_seed", result.get("status") or f"실패: {result.get('error')}"))

show_table([{"순서": number, "호출": call, "결과": outcome} for number, (call, outcome) in enumerate(history, start=1)])

print()
print("--- 서버 SQLite의 events 표 ---")
with closing(sqlite3.connect(state_dir / "sessions.sqlite3")) as database:
    stored = database.execute("select sequence, kind, payload from events order by sequence").fetchall()
show_table([{"sequence": seq, "kind": kind, "payload": payload} for seq, kind, payload in stored])

print()
print("--- 근거 폴더의 이벤트 파일 ---")
show_events(state_dir / "evidence")

**관찰할 것.** (1) `ANSWER_BOTH = True`일 때 `status`가 `draft`, `draft`, `ready`, `frozen` 순서로 바뀌는지 보세요. `ready`는 필수 답변이 모두 모였다는 뜻이고 내용이 충분하다는 뜻이 아닙니다(`skills/requirements-interview/SKILL.md:30`). (2) `False`일 때 `freeze_seed`는 예외가 아니라 `success=false`와 `seed_incomplete` 같은 **구조화된 오류 코드**로 답합니다(`cap_lab/server/workflow.py:222-227`). 이 코드가 4절의 재시도 규칙에서 쓰입니다. (3) SQLite의 `events` 줄 수와 근거 폴더의 이벤트 파일 수가 같은지 세어 보세요.

### 2.4 실행 제어: 모델과 도구 사이에 끼는 코드

실행 제어는 모델이 내놓은 도구 호출이 실제로 도구에 닿기 전후에 끼어드는 코드입니다. 이 사례에는 세 가지가 있습니다.

1. **GoalGuard** (`cap_lab/middleware.py:52`): `start_interview` 호출만 가로채서 `goal` 인자가 사용자가 말한 목표를 그대로 담았는지 Jev에게 묻고, 문턱값 아래면 도구를 부르지 않고 `goal_changed`를 돌려줍니다. 이 노트북에서는 켜지 않습니다(`gate_h2=False`가 기본). 측정은 07에서 합니다.
2. **도구 공개 지연** (`cap_lab/agent.py:148`): MCP 도구의 `defer_loading`을 켜서, SKILL.md를 읽기 전에는 모델이 스키마를 볼 수 없게 합니다.
3. **쓰기 금지** (`cap_lab/agent.py:137`): `FilesystemPermission`으로 모든 경로의 쓰기를 거절합니다. 에이전트가 Skill 폴더를 고치지 못합니다.

아래 셀은 두 코드를 **소스 그대로** 보여 줍니다. 길지 않으니 한 줄씩 읽어 보세요.

In [ ]:
# [따라하기] 실행 제어 코드를 파일:줄 번호와 함께 읽는다.
def show_source(obj) -> None:
    """함수나 메서드의 소스를 '파일:시작줄' 머리말과 줄 번호를 붙여 출력한다."""
    lines, first_number = inspect.getsourcelines(obj)
    source_file = Path(inspect.getsourcefile(obj)).relative_to(ROOT)
    print(f"=== {source_file}:{first_number} ===")
    for offset, line in enumerate(lines):
        print(f"{first_number + offset:>4}  {line.rstrip()}")
    print()


show_source(middleware_module.GoalGuard.wrap_tool_call)   # 가로챌 도구를 이름으로 고른다
show_source(agent_module._deferred)                       # 도구 사본에 defer_loading 표시를 붙인다

## 3. `create_deep_agent`의 기본 도구와 `build_agent`가 바꾸는 것

`create_deep_agent`는 도구를 따로 주지 않아도 파일 도구와 하위 에이전트 도구를 기본으로 갖춥니다. 이 과정의 에이전트는 그중 **일부를 일부러 빼고** MCP 도구 7개를 얹습니다. 무엇이 남고 무엇이 빠졌는지는 설명문이 아니라 **코드에서 읽어야** 합니다. 설치된 버전이 바뀌면 목록도 바뀌기 때문입니다.

**먼저 예측하세요.** 기본 `create_deep_agent`의 도구 이름을 아는 만큼 적어 보세요. 그다음 `build_agent`로 조립한 에이전트의 도구 목록에서 무엇이 빠졌을지, 그리고 MCP 도구 7개가 모델에게 처음부터 보일지도 적어 보세요.

### 3.1 기본 `create_deep_agent`

다음 셀은 도구 없이 `create_deep_agent`를 만들고 도구 목록을 읽습니다. 에이전트를 **만들 뿐 실행하지 않으므로 모델을 부르지 않습니다.** 그래프의 `tools` 노드가 들고 있는 도구 사전을 들여다보는 방식은 내부 구조를 엿보는 것이라 수업용 확인 방법입니다.

In [ ]:
# [따라하기] 기본 create_deep_agent가 갖는 도구 (만들기만 하고 실행하지 않는다)
bare_agent = create_deep_agent(model=make_model(settings))
bare_tools = list(bare_agent.nodes["tools"].bound.tools_by_name)

print("기본 도구:", bare_tools)
print("write_todos가 있는가:", "write_todos" in bare_tools)
if "task" not in bare_tools:
    # build_agent가 하위 에이전트를 끄는 프로필을 프로세스 전체에 등록한다 (cap_lab/agent.py:165-175).
    print("task가 없다면, 이 커널에서 build_agent를 이미 실행했기 때문이다. 커널을 다시 시작하면 보인다.")

설계 문서(`docs/DESIGN.md` 5절)는 기본 도구를 `ls, read_file, write_file, edit_file, delete, glob, grep, task`로 적고 `write_todos`는 선택이라고 적었습니다. 설치된 deepagents(0.7.32)에서 위 셀을 돌려 보면 목록에 **`execute`가 하나 더** 있고 `write_todos`는 없습니다. 문서와 코드가 다르면 코드가 정답입니다. 이 노트북은 `execute`를 부르지 않습니다. 본인의 출력이 여기 적은 것과 다르면 설치된 버전이 다른 것이니 그 차이를 기록해 두세요.

### 3.2 `build_agent`로 조립하기

`build_agent`(`cap_lab/agent.py:89`)는 다음을 순서대로 합니다.

1. `ChatAnthropic` 모델을 만든다. 키가 없으면 여기서 멈춘다(`:108`).
2. MCP 서버를 자식 프로세스로 띄우고 `tools/list`를 읽는다(`:110-114`).
3. 모든 MCP 도구에 `defer_loading` 표시를 붙인다(`:115-116`).
4. 하위 에이전트(`task`)를 끄는 프로필을 등록한다(`:117`, `:165-175`).
5. `FilesystemMiddleware(tools=["ls", "read_file", "glob", "grep"])`로 파일 도구를 읽기 전용 4개로 줄이고(`:46`, `:126-131`), Skill 폴더를 `skills=["/"]`로 연결한다(`:136`).
6. `create_deep_agent`를 부른다(`:119`). 이때도 **모델 호출은 없다.**

즉 `build_agent`가 하는 외부 호출은 MCP 서버 기동과 `tools/list`뿐이고, Claude도 Jev도 부르지 않습니다. 아래 셀에서 직접 확인합니다. 이 셀은 `build_agent`를 부르므로 규모 고지(`announce`)를 달았고, Claude도 Jev도 부르지 않으므로 고지 줄에 `유료 호출 없음`이 찍힙니다.

In [ ]:
# [따라하기] build_agent로 조립하고 모델이 쓸 도구를 읽는다.
announce("에이전트 조립만 한다", model_calls=0)
print("-> 이 셀은 Claude도 Jev도 부르지 않는다. MCP 서버 자식 프로세스만 띄웠다가 끝낸다.")

agent_dir = new_workdir("nb01-agent")
handle = await build_agent(
    settings,
    server_args=server_argv(agent_dir / "sessions.sqlite3", agent_dir / "evidence"),
    skills_dir=SKILLS_DIR,
    checkpointer=InMemorySaver(),   # MRTR 재개와 대화 이어가기에 필요하다 (cap_lab/agent.py:104)
)
try:
    built_tools = list(handle.agent.nodes["tools"].bound.tools_by_name)
    mcp_names = [tool.name for tool in handle.tools]
    deferred_names = [tool.name for tool in handle.tools if (tool.extras or {}).get("defer_loading")]
    checkpointer_name = type(handle.agent.checkpointer).__name__
    goal_guard_on = handle.goal_guard is not None
finally:
    await handle.aclose()   # stdio 자식 프로세스를 끝낸다

table = []
for name in built_tools:
    table.append({
        "도구": name,
        "출처": "MCP 서버" if name in mcp_names else "Deep Agent 파일 도구",
        "SKILL.md를 읽기 전까지 숨김": name in deferred_names,
    })
show_table(table)

print("묶여 있는 도구 수:", len(built_tools), "(파일 도구", len(built_tools) - len(mcp_names), "+ MCP", len(mcp_names), ")")
print("기본 도구 중 빠진 것:", sorted(set(bare_tools) - set(built_tools)))
print("체크포인터:", checkpointer_name)
print("GoalGuard가 붙었는가:", goal_guard_on)

**관찰할 것.** (1) 빠진 도구 목록에 쓰기 계열(`write_file`, `edit_file`, `delete`), `execute`, `task`가 들어 있는지 보세요. 각각 `cap_lab/agent.py:46`(파일 도구 4개만 허용)과 `:165`(하위 에이전트 끄기)이 한 일입니다. (2) MCP 도구 7개가 모두 `숨김 = True`인지 보세요. (3) `GoalGuard가 붙었는가`가 `False`인 것은 `gate_h2` 기본값이 `False`이기 때문입니다(`cap_lab/agent.py:94`). 이 노트북의 에이전트는 **게이트가 없는 기준선**입니다.

### 3.3 "묶여 있다"와 "모델에게 보인다"는 다르다

위 표의 도구 11개는 모두 그래프에 **묶여 있습니다.** 그러나 `숨김 = True`인 7개는 모델에게 스키마가 가지 않습니다. deepagents의 설명은 이렇습니다: 도구가 `defer_loading`이고 어떤 Skill의 `include_tools`에 이름이 있으면, 모델이 그 Skill을 `read_file`로 읽을 때 공개됩니다. 읽기 전에 부르면 **알 수 없는 도구**로 실패합니다(`.venv/lib/python3.13/site-packages/deepagents/middleware/skills.py:982-989`).

이 말은 곧, **Skill 파일이 도구 공개의 열쇠**라는 뜻입니다. `SKILL.md`를 읽지 않으면 이 에이전트는 서버의 도구를 하나도 못 씁니다. 다음 셀은 "이 Skill 폴더를 읽으면 어떤 도구가 공개되는가"를 이름 대조로 계산합니다. deepagents의 내부 계산을 그대로 부르는 것이 아니라 위 설명을 코드로 옮긴 **수업용 계산**입니다.

In [ ]:
# [따라하기] Skill 폴더가 공개할 수 있는 지연 도구를 이름 대조로 계산한다.
def disclosable_tools(skills_dir, tools) -> list[str]:
    """skills_dir 아래 모든 SKILL.md의 include_tools와, defer_loading인 도구 이름의 교집합을 돌려준다.

    deepagents가 SKILL.md를 읽은 뒤 공개하는 도구와 같은 규칙을 이름 대조로만 옮긴 수업용 계산이다.
    """
    deferred = {tool.name for tool in tools if (tool.extras or {}).get("defer_loading")}
    named = set()
    for skill_file in sorted(Path(skills_dir).rglob("SKILL.md")):
        parts = skill_file.read_text(encoding="utf-8").split("---", 2)
        if len(parts) < 3:
            continue   # 머리말이 없는 파일
        metadata = (yaml.safe_load(parts[1]) or {}).get("metadata") or {}
        named |= set(str(metadata.get("include_tools") or "").split())
    return sorted(named & deferred)


real_disclosable = disclosable_tools(SKILLS_DIR, handle.tools)
print("이 Skill 폴더를 읽으면 공개되는 MCP 도구:", real_disclosable)
print("개수:", len(real_disclosable), "/ 지연된 MCP 도구:", len(deferred_names))

### 3.4 일부러 깨뜨리기: Skill 폴더를 비우면

네 요소 중 지침을 뺍니다. Skill 폴더가 비어 있으면 도구는 **그대로 묶여 있는데** 모델은 하나도 볼 수 없게 됩니다. 오류도 나지 않습니다. 다음 셀은 비어 있는 폴더로 조립해 이것을 오프라인으로 보여 줍니다.

**먼저 예측하세요.** 묶여 있는 도구의 개수는 7개 줄어들까요, 그대로일까요? 모델이 볼 수 있는 MCP 도구는 몇 개가 될까요?

In [ ]:
# [깨뜨리기] 빈 Skill 폴더로 조립한다: 도구는 묶여 있는데 아무것도 공개되지 않는다.
announce("빈 Skill 폴더로 조립만 한다", model_calls=0)
print("-> 이 셀은 Claude도 Jev도 부르지 않는다.")

broken_dir = new_workdir("nb01-broken")
empty_skills = broken_dir / "empty-skills"
empty_skills.mkdir()

broken = await build_agent(
    settings,
    server_args=server_argv(broken_dir / "sessions.sqlite3", broken_dir / "evidence"),
    skills_dir=empty_skills,
)
try:
    broken_tools = list(broken.agent.nodes["tools"].bound.tools_by_name)
    broken_disclosable = disclosable_tools(empty_skills, broken.tools)
finally:
    await broken.aclose()

print("묶여 있는 도구 수:", len(broken_tools), "(정상 조립은", len(built_tools), "개)")
print("정상 조립과 같은 도구 이름인가:", set(broken_tools) == set(built_tools))
print("Skill을 읽어서 공개되는 MCP 도구:", broken_disclosable)

**관찰할 것.** (1) 묶여 있는 도구 수가 정상 조립과 **같은** 것을 확인하세요. 도구 목록만 보고 "잘 연결됐다"고 판단하면 이 결함을 놓칩니다. (2) 공개되는 MCP 도구가 `[]`입니다. 이 에이전트는 사용자가 인터뷰를 요청해도 `start_interview`를 부를 수 없고, 에러 없이 도구 없이 대답합니다. (3) 이 결함이 07의 단계 `S1`(Skill 트리거)에서 어떻게 보일지 생각해 보세요: `trigger_ok`가 `False`가 됩니다.

## 4. 판단은 어디에 두는가

에이전트 시스템에서 "판단"은 세 곳 중 한 곳에서 일어납니다.

| 판단 위치 | 언제 | 이 사례의 예 | 위치 |
|---|---|---|---|
| **코드 규칙** | 입력이 닫힌 값(코드, 열거형, 숫자)이고 규칙이 문장으로 쓸 수 있을 때 | 오류 코드로 재시도 여부 정하기 | `cap_lab/server/server.py:74` 등 |
| **판단 모델(Jev)** | 의미를 읽고 참/거짓이나 단계를 정해야 하고, 문턱값을 라벨로 잴 수 있을 때 | 완료 기준이 실행 결과로 확인 가능한가(`testability`) | `cap_lab/server/gate.py:28` |
| **생성(LLM)** | 결과가 문장 자체일 때 | 다음 인터뷰 질문을 어떤 말투로 건넬지 | 에이전트 모델 |

판단 기준으로 쓰는 세 가지 질문입니다.

1. 입력이 **닫힌 집합**인가? 그렇다면 코드 규칙입니다. 모델을 부를수록 느리고, 비싸고, 흔들립니다.
2. 결과가 **참/거짓이나 단계**로 줄어드는가? 그렇다면 판단 모델입니다. 확률이나 기댓값을 받아 **문턱값과 코드로 비교**합니다.
3. 결과가 **사람에게 보여 줄 문장**인가? 그렇다면 생성입니다.

그리고 한 판단을 잘못 둔 비용은 다릅니다. 코드 규칙으로 둘 것을 LLM에게 맡기면 같은 오류에 매번 다르게 반응합니다. 의미 판단을 정규식으로 쓰면 반례가 계속 나옵니다. 판단 모델의 출력을 `==`로 비교하면 5절 이전에 깨집니다.

### 4.1 코드 규칙: 오류 코드로 재시도 라우팅

서버는 업무 오류를 `success=false`와 **오류 코드**가 있는 JSON으로 돌려줍니다(`cap_lab/server/workflow.py:27-32`의 `WorkflowError`). 장애 주입(`--fault`)이 만드는 `permission_denied`와 `temporarily_unavailable`도 같은 모양입니다(`cap_lab/server/server.py:90-102`). 코드는 닫힌 집합이므로, "이 코드면 이렇게 행동한다"는 **표 하나**로 충분합니다. 이 사례에서는 그 표가 SKILL.md 본문 8번(`skills/requirements-interview/SKILL.md:41-47`)에 문장으로 적혀 있고, scorer가 결과를 코드로 확인합니다(`cap_lab/scorers.py:240`의 `no_blind_retry`, `:252`의 `retry_bounded`). 설계 문서도 이 재시도 라우팅을 "Jev 판단이 아니라 코드 규칙"인 교육용 반례로 남겨 두었습니다(`docs/DESIGN.md` P2).

다음 셀은 그 표를 **코드로** 옮깁니다. 이 함수는 수업용이며 에이전트가 실제로 부르지는 않습니다. 실제 에이전트는 SKILL.md의 문장을 읽고 행동하고, 그 행동이 규칙을 지켰는지는 scorer가 확인합니다.

In [ ]:
# [따라하기] 구조화된 오류 코드 -> 행동 표 (코드 규칙의 모양)
RETRY_RULES = {
    # 아래 네 줄은 SKILL.md 41~47줄의 내용이다.
    "goal_changed": "호출이 실행되지 않았다. 목표 문장을 그대로 넣어 다시 호출한다",
    "permission_denied": "같은 호출을 반복하지 않는다. 거절된 도구를 사용자에게 알린다",
    "temporarily_unavailable": "같은 인자로 한 번만 다시 보낸다. 또 실패하면 멈추고 알린다",
    "evidence_publish_failed": "DB 저장 여부를 알리고 get_session으로 복구한다",
    # 이 줄은 수업용 예다. 서버가 주는 코드(cap_lab/server/workflow.py:222-227)에 대한 행동을 우리가 정했다.
    "seed_incomplete": "남은 질문에 답을 받은 뒤에 freeze_seed를 다시 호출한다",
}


def route_by_error_code(result: dict, rules: dict = RETRY_RULES) -> str:
    """도구 결과 하나를 보고 다음 행동을 돌려준다. 성공이면 계속하고, 모르는 코드는 사람에게 넘긴다."""
    if result.get("success") is not False:
        return "계속 진행한다"
    return rules.get(result.get("error"), f"모르는 오류 코드({result.get('error')}): 사람에게 넘긴다")


samples = [
    WorkflowError("temporarily_unavailable", "일시적으로 처리할 수 없습니다.", retryable=True).result,
    WorkflowError("permission_denied", "이 도구를 사용할 권한이 없습니다.").result,
    WorkflowError("seed_incomplete", "필수 질문에 답한 뒤 명세를 확정하세요.", missing_fields=["constraints"]).result,
    {"success": True, "status": "ready"},
]
show_table([{"결과의 error": sample.get("error", "-"), "다음 행동": route_by_error_code(sample)} for sample in samples])

# 서버가 이 딕셔너리를 tools/call 결과로 싸서 보내는 모양 (cap_lab/server/server.py:74)
wire = business_failure(samples[0])
print()
print("tools/call 결과의 최상위 키:", sorted(wire))
print("structuredContent의 error:", wire["structuredContent"]["error"])
print("장애 주입 계획의 모양(FaultPlan):", FaultPlan(permission_denied=["freeze_seed"], transient={"record_answer": 1}).model_dump())

In [ ]:
# [바꿔보기] 표에 없는 오류 코드가 오면? 규칙을 한 줄 추가해 본다.
NEW_CODE = "rate_limited"   # 다른 코드로 바꿔 본다

unknown_result = {"success": False, "error": NEW_CODE, "message": "요청이 너무 많습니다."}
print("규칙에 없을 때:", route_by_error_code(unknown_result))

my_rules = {**RETRY_RULES, NEW_CODE: "잠시 기다린 뒤 한 번만 다시 보낸다"}   # 한 줄 추가
print("규칙을 추가한 뒤:", route_by_error_code(unknown_result, my_rules))

# 규칙 표는 닫힌 집합이므로 빠진 코드는 눈에 띄게 실패한다(사람에게 넘김).
# 이 실패는 반복해도 같은 결과다. LLM이 같은 질문에 매번 다르게 답하는 것과 다르다.

### 4.2 판단 모델(Jev): 완료 기준이 테스트 가능한가

`record_answer`가 완료 기준을 받을 때, 서버는 그 문장이 **실행 결과로 참/거짓을 판정할 수 있는지**를 Jev에게 묻습니다(`--gate h1`일 때, `cap_lab/server/server.py:259`). 질문은 `Score` 하나입니다(`cap_lab/server/gate.py:28`의 `TESTABILITY`). 세 단계로 판정하게 합니다: "판정할 수 없다", "일부만 판정할 수 있다", "실행 결과로 참/거짓을 판정할 수 있다"(`cap_lab/server/gate.py:34`).

결과(`Verdict`, `cap_lab/decisions.py:57`)에서 볼 것은 세 가지입니다.

- `score`: 단계별 확률로 낸 **기댓값**입니다. 척도는 Decision API가 정하므로 `[0, 1]`이라고 가정하지 않습니다(`cap_lab/server/gate.py:3`).
- `probabilities`: 단계별 확률 분포.
- `confidence`: **분포의 모양**입니다. 확률이 한 단계에 모여 있으면 높고, 퍼져 있으면 낮습니다. **맞을 확률이 아닙니다.** 두 단계 사이에 깔끔하게 놓인 항목은 모델이 확신해도 `confidence`가 낮을 수 있습니다(`.agents/skills/langgraph-decision-models/SKILL.md:121`).

문턱값(band)과의 비교는 코드로 합니다. 서버의 `GateDecision.passed`는 `score >= threshold`입니다(`cap_lab/server/gate.py:51-54`). 기본 문턱값 0.5는 **라벨로 재지 않은 자리값**입니다(`cap_lab/server/gate.py:4`).

**먼저 예측하세요.** 세 완료 기준 `"잘 되면 완료"`, `"사용자가 만족하면 완료"`, `"pytest tests/test_rental.py가 실패 0건으로 끝나면 완료"`를 Jev가 `score`로 정렬한다면 어떤 순서일까요? `confidence`가 가장 높은 것은 어느 쪽일까요?

In [ ]:
# [따라하기] Jev에게 완료 기준 세 개의 testability를 묻는다.
CRITERIA = {
    "모호": "잘 되면 완료",
    "주관": "사용자가 만족하면 완료",
    "구체": "pytest tests/test_rental.py가 실패 0건으로 끝나면 완료",
}
THRESHOLD = 0.5   # 라벨로 재지 않은 자리값 (cap_lab/server/gate.py:4)

announce("Jev testability 판정 3건", model_calls=0, jev_calls=3)
decider = make_decider(settings)

jev_scores = {}   # 라벨 -> Verdict. 바꿔보기 셀이 다시 쓴다.
jev_rows = []
for label, criterion in CRITERIA.items():
    try:
        judgment = decider.judge(criterion, [TESTABILITY])
    except DecisionError as error:
        print(f"[{label}] 판정 실패: {error}")   # 메시지에는 공급자 이름과 HTTP 상태만 있다
        continue
    verdict = judgment.verdicts[TESTABILITY.name]
    jev_scores[label] = verdict
    passed = None if verdict.score is None else GateDecision(verdict.score, THRESHOLD).passed
    jev_rows.append({
        "라벨": label,
        "완료 기준": criterion,
        "거절됨": verdict.refused,
        "score": verdict.score,
        "confidence": verdict.confidence,
        "확률 분포": verdict.probabilities,
        f"문턱값 {THRESHOLD} 이상": passed,
        "백엔드/모델": f"{judgment.backend}/{judgment.model}",
    })
show_table(jev_rows)

**관찰할 것.** 이 결과는 **1회 관찰**입니다. (1) `score`의 순서가 예측과 같은지, 그리고 값이 `[0, 1]` 안에 있는지 보세요. 안 맞더라도 오류가 아니라 척도를 확인하는 일입니다. (2) `확률 분포`의 합과 모양을 보고 `confidence`가 그 모양과 맞는지 보세요. `confidence`가 높다고 `score`가 옳은 것은 아닙니다. (3) `문턱값 이상`이 `False`인 완료 기준이 있다면, 서버는 모델이 쓴 문장을 기록하지 않고 사용자에게 직접 되묻습니다(`cap_lab/server/server.py:242-246`). 이 되묻기는 모델이 보는 도구 인자에 나타나지 않습니다. (4) 같은 문장을 다시 판정하면 값이 조금 달라질 수 있습니다. 서버가 첫 점수를 저장해 두는 이유입니다(`cap_lab/server/gate.py:63-65`).

In [ ]:
# [바꿔보기] 문턱값을 움직이면 같은 점수의 합격 여부가 어떻게 달라지는가 (Jev 호출 없음)
if len(jev_scores) < 2 or any(verdict.score is None for verdict in jev_scores.values()):
    print("앞 셀에서 점수를 두 개 이상 받지 못했습니다. 앞 셀을 먼저 실행하세요.")
else:
    values = [verdict.score for verdict in jev_scores.values()]
    low, high = min(values), max(values)
    # 척도를 모르므로 관찰된 점수 범위 안에서 문턱값 세 개를 고른다.
    sweep = [low + (high - low) * fraction for fraction in (0.25, 0.5, 0.75)]

    sweep_rows = []
    for threshold in sweep:
        row = {"문턱값": round(threshold, 3)}
        for label, verdict in jev_scores.items():
            row[label] = GateDecision(verdict.score, threshold).passed
        sweep_rows.append(row)
    show_table(sweep_rows)
    print("점수 범위:", round(low, 3), "~", round(high, 3))
    print("문턱값을 올리면 통과가 줄어든다. 어느 값이 맞는지는 라벨로 재서 정한다 (docs/DESIGN.md P4).")

### 4.3 일부러 깨뜨리기: 기댓값을 `==`로 비교하면

`score`는 확률로 낸 기댓값입니다. 기댓값은 소수점 연산의 결과라서 **같은 값인지**로 비교하면 깨집니다. 아래 셀은 모델 호출 없이 이것을 보여 줍니다. 단계 위치를 `0, 1, 2`로 두고 확률 `0.1, 0.2, 0.7`로 기댓값을 계산하는 것은 **수업용 가정**입니다. 실제 Jev의 척도는 Decision API가 정합니다(`cap_lab/server/gate.py:3`).

In [ ]:
# [깨뜨리기] 기댓값을 ==로 비교한다 (모델 호출 없음)
probabilities = [0.1, 0.2, 0.7]   # 수업용 가정: 단계 0, 1, 2의 확률
expected = sum(position * probability for position, probability in enumerate(probabilities))
print("기댓값:", repr(expected))
print()
print("expected == 1.6        :", expected == 1.6, "  <- 손으로 계산하면 1.6인데?")
print("expected >= 1.6        :", GateDecision(expected, 1.6).passed, "  <- 경계값에서는 >=도 흔들린다")
print("expected >= 1.5        :", GateDecision(expected, 1.5).passed, "  <- 경계에서 떨어진 문턱값")
print("math.isclose(expected, 1.6):", math.isclose(expected, 1.6))

**관찰할 것.** (1) 손으로 계산하면 `1.6`인 값이 `== 1.6`에서 `False`입니다. `==`로 쓴 band는 **조용히** 틀립니다. 오류도 경고도 없습니다. (2) 놀라운 점은 `>= 1.6`도 `False`라는 것입니다. 문턱값이 계산된 값과 정확히 같은 경계에 놓이면 `>=`도 오차에 흔들립니다. 그래서 문턱값은 **라벨로 재어서** 값이 몰려 있지 않은 곳에 잡고, 정말 같은 값인지 알아야 할 때는 `math.isclose`를 씁니다. (3) 서버 코드가 `<`와 `>=`만 쓰고 `==`를 쓰지 않는 이유가 이것입니다(`cap_lab/server/gate.py:4`, `:54`). 확률(`Noul`)도 같습니다. `GoalGuard`는 `probability >= threshold`로 비교합니다(`cap_lab/middleware.py:85`).

### 4.4 생성(LLM)과 그 곁에 붙는 판단

다음 인터뷰 질문의 말투, 사용자의 답을 `record_answer`로 옮겨 적는 일, 최종 요약문을 쓰는 일은 **생성**입니다. 표로 줄일 수 없고 정답 하나가 없습니다. 이런 일에 판단 모델을 쓸 이유가 없습니다.

다만 생성이 만든 **도구 인자**는 다릅니다. 모델이 `start_interview(goal=...)`의 `goal`에 사용자 문장을 "다듬어서" 넣으면 목표가 바뀝니다. 이때 세 판단 위치가 한 줄로 이어집니다.

1. LLM이 `goal` 인자를 **생성**한다.
2. `GoalGuard`가 Jev의 `Noul` 질문 `goal_preserved`로 **판단**한다(`cap_lab/middleware.py:26`).
3. `확률 >= 문턱값`인지는 **코드**가 정한다(`cap_lab/middleware.py:85`). 판정을 못 하면 도구를 부르지 않는다(fail-closed, `:88-90`).

다음 셀은 그 질문 문장을 읽습니다. Jev를 부르지 않습니다.

In [ ]:
# [따라하기] GoalGuard가 Jev에게 묻는 질문을 읽는다 (호출 없음)
question = middleware_module.GOAL_PRESERVED
print("질문 이름:", question.name, "| 종류:", question.kind)
print()
print("instructions:", question.instructions)
print()
print("참(true)의 기준:", question.true)
print("거짓(false)의 기준:", question.false)
print()
print("막았을 때 모델에게 돌려주는 말:", middleware_module.GOAL_CHANGED_MESSAGE)
print("도구를 막는 도구 이름:", middleware_module.GUARDED_TOOL)

show_table([
    {"단계": "goal 인자를 만든다", "판단 위치": "생성(LLM)", "근거": "에이전트 모델"},
    {"단계": "goal이 사용자 말과 같은 뜻인가", "판단 위치": "판단 모델(Jev)", "근거": "cap_lab/middleware.py:26"},
    {"단계": "확률이 문턱값 이상인가", "판단 위치": "코드 규칙", "근거": "cap_lab/middleware.py:85"},
    {"단계": "막을 때 어떤 오류를 주는가", "판단 위치": "코드 규칙", "근거": "cap_lab/middleware.py:88-92"},
])

## 5. 실제 에이전트 한 번 돌려 보기

이제 앞의 네 요소를 조립한 에이전트를 **한 번** 실행합니다. 사례는 dev-01입니다. 사용자가 팀 비품 대여 현황 도구를 만들고 싶다고 말하고, 에이전트가 Skill을 읽고 `start_interview`를 부르는 것이 기대입니다.

실행기는 `cap_lab/experiment.py:131`의 `run_case`입니다. 이 함수는 사례의 `turns`(사용자 발화)만 에이전트에게 주고, 사례의 `expect`(기대)는 점수 계산에만 씁니다(`cap_lab/experiment.py:2-4`의 설명). 한 번의 실행이 끝나면 `RunRecord`에 아래가 쌓입니다(`cap_lab/scorers.py:42-55`).

- `tool_calls`: 도구 호출을 호출 순서대로. 이름, 인자, 결과, 성공 여부.
- `skill_reads`: `read_file`로 읽은 SKILL.md 경로(`cap_lab/experiment.py:284`).
- `model_calls`: 모델 노드가 한 번 돌 때마다 1(`cap_lab/experiment.py:246-247`).
- `input_tokens`, `output_tokens`, `latency_s`, `final_text`.

**먼저 예측하세요.** 이 한 번의 실행에서 (1) `read_file`이 `start_interview`보다 앞에 올까요, 뒤에 올까요? (2) 도구 호출은 몇 번일까요? (3) 에이전트는 사용자 대신 `record_answer`까지 부를까요, 질문만 하고 멈출까요?

In [ ]:
# [따라하기] dev-01 사례를 에이전트로 한 번 실행한다.
dev_cases = {case["id"]: case for case in load_cases(split="dev")}
live_case = dev_cases["dev-01"]
print("사용자 발화:", live_case["turns"][0])
print("사례 종류:", live_case["kind"], "| 단계:", live_case["stage"])

announce("dev-01 한 번 실행", model_calls=9, runs=1)
live_dir = new_workdir("nb01-live")
record = await run_case(
    live_case, variant=Variant(name="A"), settings=settings, workdir=live_dir, repeat=1)
print("실행 오류:", record.error or "없음")

**관찰할 것.** 이 결과는 **1회 관찰**입니다. (1) `실행 오류`가 `없음`인지 먼저 보세요. 오류가 있으면 아래 점수는 확인 불가(`None`)가 됩니다(`cap_lab/scorers.py:299-310`). (2) 오류가 없으면 다음 셀로 트레이스를 읽습니다. 같은 사례를 다시 돌리면 도구 호출 순서나 횟수가 달라질 수 있다는 점을 기억하세요. 그래서 07에서는 같은 사례를 여러 번 돌립니다.

In [ ]:
# [따라하기] RunRecord를 읽기 좋은 트레이스로 풀어 쓴다.
def print_trace(record) -> None:
    """한 번의 실행 기록에서 호출 규모, 읽은 Skill, 도구 호출 순서, 최종 답변 앞부분을 출력한다."""
    print(f"모델 호출 {record.model_calls}회 | 도구 호출 {len(record.tool_calls)}회 | "
          f"입력 토큰 {record.input_tokens} | 출력 토큰 {record.output_tokens} | 시간 {record.latency_s:.1f}초")
    print("읽은 Skill 파일:", record.skill_reads or "없음")
    print()
    print("도구 호출 순서")
    status_text = {True: "성공", False: "실패(업무 오류 또는 도구 오류)", None: "결과 없음"}
    for number, call in enumerate(record.tool_calls, start=1):
        arguments = json.dumps(call["args"], ensure_ascii=False)
        print(f"{number:>3}. {call['name']:<16} {status_text[call['ok']]:<8} 인자={arguments[:90]}")
    print()
    print("최종 답변(앞 300자):", record.final_text[:300])


print_trace(record)

In [ ]:
# [바꿔보기] 이 한 번의 실행을 단계별 점수로 읽고, 서버가 남긴 근거를 본다.
scores = score_run(record)
score_rows = [
    {"점수": score.name, "단계": STAGE_OF.get(score.name, "-"), "값": score.value, "이유": score.comment}
    for score in scores
]
show_table(score_rows)
print("첫 실패 단계:", first_failure(scores))
print()
print("--- 이 실행에서 서버가 남긴 이벤트 ---")
show_events(record.evidence_dir)

**관찰할 것.** (1) `값`이 `None`인 줄은 "이 사례에는 해당 없음"입니다. `False`(실패)로 읽으면 안 됩니다(`cap_lab/scorers.py:4`). (2) `trigger_ok`는 SKILL.md를 읽었는지, `tool_selection_ok`는 불러야 할 도구를 불렀는지입니다. 모두 파일과 호출 기록을 **세는 코드**가 계산하고 모델 판정은 쓰지 않습니다. (3) `첫 실패 단계`가 `None`이면 이 한 번에서는 실패가 없었다는 뜻입니다. 잘 동작한다는 증명이 아니라 1회 관찰입니다. (4) 근거 이벤트가 `interview_started` 하나뿐인지, 그 뒤 질문에 대한 답 기록이 있는지 확인하세요. 에이전트는 사용자 대신 답을 지어 `record_answer`를 부르면 안 됩니다(`skills/requirements-interview/SKILL.md:24-26`).

In [ ]:
# [바꿔보기] (선택, 돈이 든다) 같은 사례를 도구를 처음부터 모두 공개한 에이전트로 한 번 더 돌려 본다.
if RUN_OPTIONAL:
    announce("dev-01을 도구 처음부터 공개로 한 번 실행", model_calls=9, runs=1)
    upfront_dir = new_workdir("nb01-upfront")
    upfront_record = await run_case(
        live_case, variant=Variant(name="U", tool_disclosure="upfront"),
        settings=settings, workdir=upfront_dir, repeat=1)
    print("실행 오류:", upfront_record.error or "없음")
    print_trace(upfront_record)
    print()
    print("앞 실행의 모델 호출:", record.model_calls, "| 이 실행의 모델 호출:", upfront_record.model_calls, "(각각 1회 관찰)")
else:
    print("RUN_OPTIONAL = False 이므로 건너뜁니다. 켜면 Claude 약 5~9회를 씁니다.")

**관찰할 것.** 선택 셀을 돌렸다면 두 번의 **1회 관찰**을 비교하는 것은 증거가 아니라 질문거리입니다. 호출 수 차이가 도구 공개 방식 때문인지 실행마다의 흔들림인지 이 두 번으로는 가를 수 없습니다. 그것을 가르는 방법(같은 구성의 반복으로 잡음 폭을 재는 A/A 비교)이 07의 주제입니다.

## 6. 병목 귀속 미리보기와 정리

한 번의 실행이 실패하면 **어디서** 처음 어긋났는지를 알아야 고칠 곳이 정해집니다. 07은 실패를 가장 이른 단계에 귀속합니다(`docs/DESIGN.md` P1, P3).

| 단계 | 무엇을 보나 | scorer |
|---|---|---|
| S1 | Skill을 읽었나(트리거) | `trigger_ok` |
| S2 | 맞는 도구를 골랐고 인자가 유효하며 목표를 유지했나 | `tool_selection_ok`, `args_valid`, `goal_preserved` |
| S3 | 모호한 답을 확정하지 않았나 | `vague_not_frozen` |
| S5 | 확정 결과가 맞나 | `frozen_ok` |
| S6 | 산출물이 근거에 연결됐나 | `evidence_linked` |

단계 번호가 `S1, S2, S3, S5, S6`이고 `S4`가 없는 것은 설계 중 동의 단계(S4)를 뺐기 때문입니다(`docs/DESIGN.md` P1). `R1~R3`(권한 거부, 일시 오류, 중복 호출)은 견고성 검사이고 첫 실패 귀속에는 들어가지 않습니다(`cap_lab/scorers.py:26`, `:322-325`).

병목이 되려면 세 조건이 모두 필요합니다(P3): (1) 처음 실패하는 비중이 가장 크고, (2) 판단 모양의 문제이며, (3) 그것을 푸는 결정적 규칙이 없을 것. 지금 보는 **판단 위치 구분**이 (2)와 (3)을 가르는 도구입니다. 오류 코드로 풀리는 문제는 Jev에 가지 않습니다.

In [ ]:
# [따라하기] 단계 순서와 scorer 이름을 코드에서 읽는다.
stage_meaning = {"S1": "Skill 트리거", "S2": "시작 인자", "S3": "모호한 답변", "S5": "확정", "S6": "근거 연결"}
stage_rows = []
for stage in STAGE_ORDER:
    names = [name for name, mapped in STAGE_OF.items() if mapped == stage]
    stage_rows.append({"단계": stage, "의미": stage_meaning[stage], "scorer": ", ".join(names)})
show_table(stage_rows)
print("첫 실패는 위 순서에서 가장 이른 단계다. STAGE_ORDER =", STAGE_ORDER)
print("견고성 검사(첫 실패에 안 들어감):", [name for name, mapped in STAGE_OF.items() if mapped.startswith("R")])

In [ ]:
# [바꿔보기] 가상의 실패 이름을 넣어 첫 실패 단계가 어디로 귀속되는지 본다 (호출 없음)
FAILED = {"vague_not_frozen", "frozen_ok"}   # 이 이름들을 바꿔 가며 실행한다. 예: {"args_valid"}, {"no_blind_retry"}


def attribute(failed_names: set[str]) -> str | None:
    """STAGE_OF에 있는 모든 scorer 중 failed_names만 False로 두고 첫 실패 단계를 돌려준다."""
    what_if = [Score(name, name not in failed_names, "가상 점수") for name in STAGE_OF]
    return first_failure(what_if)


print("실패한 scorer:", sorted(FAILED))
print("귀속 단계:", attribute(FAILED))
print()
print("S3와 S5가 함께 실패하면 S3만 센다:", attribute({"vague_not_frozen", "frozen_ok"}))
print("S5만 실패하면:", attribute({"frozen_ok"}))
print("견고성 검사만 실패하면 귀속 단계가 없다:", attribute({"no_blind_retry"}))

In [ ]:
# [내 것 만들기] 본인이 맡은 에이전트(또는 떠올린 에이전트)의 판단 세 가지를 분류한다.
JUDGMENT_KINDS = ("코드 규칙", "판단 모델(Jev)", "생성(LLM)")

# 아래 세 줄을 본인의 판단으로 바꾼다. 앞의 값은 예시다.
MY_DECISIONS = {
    "주문 상태가 '배송 중'이면 취소 불가로 안내": "코드 규칙",
    "고객 문의가 환불 요청인지 단순 문의인지 가름": "판단 모델(Jev)",
    "사과와 안내를 담은 답장 문장 쓰기": "생성(LLM)",
}

problems = [text for text, kind in MY_DECISIONS.items() if kind not in JUDGMENT_KINDS]
if problems:
    print("분류 이름이 틀린 항목:", problems, "| 허용:", JUDGMENT_KINDS)
else:
    show_table([{"판단": text, "위치": kind} for text, kind in MY_DECISIONS.items()])
    for kind in JUDGMENT_KINDS:
        print(f"{kind}: {sum(1 for value in MY_DECISIONS.values() if value == kind)}개")

**셀을 돌린 뒤 스스로 물을 것.** 세 항목 중 코드 규칙으로 둔 것은 입력이 정말 닫힌 집합인가요? 판단 모델로 둔 것은 문턱값을 재어 줄 **라벨**을 구할 수 있나요? 라벨을 구할 수 없다면 그 판단은 아직 판단 모델에 맡길 준비가 되지 않은 것입니다.

## 직접 해보기

### 해보기 1. 기대가 반대인 사례를 돌려 보기 (돈이 듭니다)

dev-02는 MCP와 Seed가 무엇인지 묻는 개념 질문입니다. 기대는 Skill을 **읽지 않고** `start_interview`도 부르지 않는 것입니다(`skills/requirements-interview/SKILL.md:15`의 `MCP나 Seed의 개념만 묻는 질문에는 ... 쓰지 않는다`). 다음 셀의 `RUN_EXERCISE`를 `True`로 바꾸기 전에 예측부터 적으세요: 에이전트가 이 질문에 Skill을 읽을까요? 읽었다면 어느 단계에서 실패한 것으로 귀속될까요?

In [ ]:
# [바꿔보기] dev-02(개념 질문)를 한 번 실행한다. RUN_EXERCISE를 맨 위 셀에서 True로 바꿔야 돈이 든다.
if RUN_EXERCISE:
    ex_case = dev_cases["dev-02"]
    print("사용자 발화:", ex_case["turns"][0])
    announce("dev-02 한 번 실행", model_calls=9, runs=1)
    ex_dir = new_workdir("nb01-ex1")
    ex_record = await run_case(
        ex_case, variant=Variant(name="A"), settings=settings, workdir=ex_dir, repeat=1)
    print("실행 오류:", ex_record.error or "없음")
    print_trace(ex_record)
    ex_scores = score_run(ex_record)
    print("첫 실패 단계:", first_failure(ex_scores), "(1회 관찰)")
else:
    print("RUN_EXERCISE = False 이므로 건너뜁니다. 켜면 Claude 약 5~9회를 씁니다.")

### 해보기 2. Skill의 `include_tools`를 줄여 보기 (돈이 들지 않습니다)

Skill 폴더 사본을 임시 폴더에 만들고 `include_tools`를 두 개로 줄이면, 어떤 MCP 도구가 공개되고 어떤 도구는 영영 공개되지 않을까요? 원본은 건드리지 않습니다. 예측을 적은 뒤 셀을 돌리세요.

In [ ]:
# [내 것 만들기] 임시 폴더에 Skill 사본을 만들고 include_tools를 바꿔 공개되는 도구를 계산한다.
KEEP_TOOLS = ["start_interview", "record_answer"]   # 본인이 고른 도구 이름으로 바꾼다

copy_root = Path(tempfile.mkdtemp(prefix="nb01-skill-copy-"))
copy_dir = copy_root / "requirements-interview"
copy_dir.mkdir()

original_text = skill_path.read_text(encoding="utf-8")
head, body = original_text.split("---", 2)[1], original_text.split("---", 2)[2]
edited_head = yaml.safe_load(head)
edited_head["metadata"]["include_tools"] = " ".join(KEEP_TOOLS)
(copy_dir / "SKILL.md").write_text("---\n" + yaml.safe_dump(edited_head, allow_unicode=True) + "---" + body, encoding="utf-8")

shown = disclosable_tools(copy_root, handle.tools)
hidden_forever = sorted(set(deferred_names) - set(shown))
print("읽으면 공개되는 도구:", shown)
print("이 Skill로는 영영 공개되지 않는 도구:", hidden_forever)
print("원본 Skill은 그대로인가:", skill_path.read_text(encoding="utf-8") == original_text)

**생각해 볼 것.** 공개되지 않는 도구를 모델이 부르려 하면 어떤 일이 생기나요(3.3절)? 그 결과 07의 어느 단계 점수가 `False`가 될까요?

### 해보기 3. 내 완료 기준 세 개를 Jev에 물어보기 (돈이 듭니다)

4.2절의 `CRITERIA`를 본인의 완료 기준 세 개로 바꿔 판정합니다. 하나는 일부러 모호하게, 하나는 일부러 구체적으로 쓰세요. 결과가 예측과 다르면 어느 쪽이 틀렸는지(질문 문구, 기준 문장, 척도 가정)를 따져 보세요.

In [ ]:
# [내 것 만들기] 본인의 완료 기준 세 개를 Jev로 판정한다. RUN_EXERCISE를 True로 바꿔야 돈이 든다.
MY_CRITERIA = {
    "모호": "사용자가 편하게 쓰면 완료",
    "중간": "대여 현황 화면이 나오면 완료",
    "구체": "대여 목록 API가 200을 돌려주고 항목 수가 DB와 같으면 완료",
}

if RUN_EXERCISE:
    announce("내 완료 기준 3건 판정", model_calls=0, jev_calls=3)
    my_decider = make_decider(settings)
    my_rows = []
    for label, criterion in MY_CRITERIA.items():
        try:
            my_verdict = my_decider.judge(criterion, [TESTABILITY]).verdicts[TESTABILITY.name]
        except DecisionError as error:
            print(f"[{label}] 판정 실패: {error}")
            continue
        my_rows.append({"라벨": label, "score": my_verdict.score, "confidence": my_verdict.confidence,
                        "확률 분포": my_verdict.probabilities})
    show_table(my_rows)
else:
    print("RUN_EXERCISE = False 이므로 건너뜁니다. 켜면 Jev 최대 3회를 씁니다.")

## 확인 질문

1. 능력을 이루는 네 요소를 이 사례의 파일 이름으로 하나씩 대 보세요. 그중 하나를 뺐을 때 **오류 없이** 도구가 쓸모없어지는 경우는 무엇이었나요?
2. `build_agent`로 조립한 에이전트의 도구가 11개인데 모델에게 처음 보이는 것은 몇 개인가요? 나머지가 보이게 하는 사건은 무엇이고, 그 근거는 어느 파일에 있나요?
3. "`temporarily_unavailable`이면 한 번만 다시 보낸다"를 Jev로 판단하지 않고 코드 규칙으로 두는 이유를, 입력의 성질로 설명해 보세요.
4. Jev의 `score`를 `== 문턱값`으로 비교하면 왜 조용히 틀리나요? `>=`로 바꿔도 남는 위험은 무엇인가요?
5. `confidence`가 높은 판정과 맞는 판정이 같은 말이 아닌 이유를 한 문장으로 말해 보세요.
6. 한 번의 실행에서 `first_failure`가 `None`이었습니다. 이 에이전트가 잘 동작한다고 말할 수 있나요? 무엇이 더 있어야 하나요?

## 정리

- **능력 = 지침 + 도구 + 실행 제어 + 상태.** 하나가 빠져도 오류 없이 조용히 무력해질 수 있습니다. 빈 Skill 폴더가 그 예였습니다(3.4절).
- **묶여 있음과 보임은 다릅니다.** `build_agent`는 파일 도구 4개와 MCP 도구 7개를 묶고, MCP 도구는 SKILL.md를 읽을 때까지 숨깁니다(`cap_lab/agent.py:46`, `:115-116`).
- **판단 위치 세 가지.** 닫힌 집합이면 코드 규칙(오류 코드 라우팅), 의미를 읽어 단계나 참/거짓으로 줄이면 판단 모델(Jev의 testability와 goal_preserved), 문장 자체면 생성(LLM). 판단 모델의 출력은 `==`가 아니라 문턱값과 `<`/`>=`로 비교하고, 문턱값은 라벨로 잽니다.
- **한 번의 실행은 1회 관찰입니다.** 읽기 좋은 트레이스는 질문을 만드는 데 쓰고, 효과가 있는지는 반복과 비교로 확인합니다.
- **실패는 가장 이른 단계에 귀속합니다.** `S1 -> S2 -> S3 -> S5 -> S6`의 순서이고 견고성 검사 `R1~R3`은 따로 봅니다.

다음 노트북 `02-Tool-Interface`에서는 모델이 도구를 정확히 고르고 인자를 채우게 하려면 도구의 이름, 설명, 인자 형식을 어떻게 써야 하는지 같은 서버로 확인합니다.